# What actually wins on the ladder, read from the battle logs

Most meta reads on this competition hand you a table of numbers and ask you to trust them. This one
recomputes the whole thing in front of you from the raw episode logs, so every archetype share,
matchup, and win rate below comes from parsing the actual games, not from a pasted spreadsheet. Fork
it, point it at a different day, and it recomputes.

The plan is simple. Each episode is one ranked battle between two submitted agents. From its log I
read what each side actually played, label each side by its ace ex-Pokemon, and read the winner from
the final rewards. Aggregate that over a day of ladder games and you get the real archetype tier
list, the matchup grid, which decks end games fast, a check of whether the crowd is actually playing
the decks that win, and a single answer to the only question that matters at submission time: which
deck to bring into this field.

In [ ]:
import os, glob, json, re, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
INK = "#1b2a4a"; ACC = "#c1440e"; ACC2 = "#1f6f8b"; MUT = "#8a94a6"

def find_one(names):
    for n in names:
        h = glob.glob(f"/kaggle/input/**/{n}", recursive=True) + glob.glob(f"D:/kaggle/**/{n}", recursive=True)
        if h: return h[0]
    return None

card_path = find_one(["EN_Card_Data.csv"])
cd = pd.read_csv(card_path)
idc = "Card ID"; namec = "Card Name"
cd["nm"] = cd[namec].astype(str)
cat = cd["Category"].astype(str).str.lower() if "Category" in cd else pd.Series("", index=cd.index)
hp = pd.to_numeric(cd["HP"], errors="coerce").fillna(0).values if "HP" in cd else np.zeros(len(cd))
id2name = dict(zip(cd[idc], cd["nm"]))
id2hp = dict(zip(cd[idc], hp))
# a card is an ace if it is a Pokemon whose name carries the ex/EX rule marker
is_ace = {}
for cid, nm, hpv in zip(cd[idc], cd["nm"], hp):
    is_ace[cid] = bool(re.search(r"\bex\b", str(nm), re.I)) and hpv > 0
print("card data:", card_path, "|", len(cd), "cards |", sum(is_ace.values()), "ace ex-Pokemon")

In [ ]:
# locate a day of episode logs (each *.json is one ranked battle)
epdir = None
for d in sorted(glob.glob("/kaggle/input/*")) + ["D:/kaggle/scratchpad/poke_ep"]:
    js = [p for p in glob.glob(os.path.join(d, "*.json")) if os.path.basename(p)[:-5].isdigit()]
    if not js:  # the dump may nest its jsons one level down; look deeper before moving on
        js = [p for p in glob.glob(os.path.join(d, "**", "*.json"), recursive=True)
              if os.path.basename(p)[:-5].isdigit()]
    if js:
        epdir = os.path.dirname(js[0]); break
assert epdir is not None, "no episode json folder found under /kaggle/input"
files = sorted(p for p in glob.glob(os.path.join(epdir, "*.json")) if os.path.basename(p)[:-5].isdigit())
CAP = 4000                      # tractability cap; disclosed in Honest scope at the end
if len(files) > CAP:
    import random
    random.seed(0)
    files = sorted(random.sample(files, CAP))   # random sample of the day, not the earliest slice
print("episode dir:", epdir, "| battles to parse:", len(files))

In [ ]:
def archetype(cards_played):
    # ace = highest-HP ex-Pokemon the side actually played; sides with no ex go to a catch-all bucket
    aces = [(id2hp.get(c, 0), c) for c in cards_played if is_ace.get(c)]
    if aces:
        return id2name.get(max(aces)[1], "?")
    return "no ex ace"

def parse_episode(path):
    try:
        d = json.load(open(path, encoding="utf-8"))
    except Exception:
        return None
    rew = d.get("rewards") or []
    if len(rew) != 2 or rew[0] is None or rew[1] is None or rew[0] == rew[1]:
        return None                       # ties, crashed games and malformed logs are dropped
    winner = 0 if rew[0] > rew[1] else 1
    played = {0: [], 1: []}
    n_steps = len(d.get("steps") or [])
    # each seat's observation is its own information set; read player i's card events from
    # player i's view, or seat 1's private plays are invisible and it gets mislabeled
    for st in d.get("steps", []):
        if not isinstance(st, list):
            continue
        for i in range(min(2, len(st))):
            obs = st[i].get("observation", {}) if isinstance(st[i], dict) else {}
            for lg in obs.get("logs") or []:
                if lg.get("playerIndex") == i and lg.get("cardId") is not None:
                    played[i].append(lg["cardId"])
    return dict(a0=archetype(played[0]), a1=archetype(played[1]), winner=winner, n_steps=n_steps)

t0 = time.time(); rows = []
for k, p in enumerate(files, 1):
    r = parse_episode(p)
    if r: rows.append(r)
    if k % 500 == 0:
        print(f"  {k}/{len(files)} files, {len(rows)} decisive, {round(time.time()-t0)}s", flush=True)
ep = pd.DataFrame(rows)
print(f"parsed {len(ep)} decisive battles of {len(files)} files in {round(time.time()-t0)}s "
      f"({len(files)-len(ep)} dropped as tie/crash/malformed)")
print("distinct archetypes:", ep[["a0", "a1"]].stack().nunique())

## 1. Who is actually being played

Every battle has two sides; here is how often each ace shows up across all of them. This is raw
presence on the ladder, the denominator for everything that follows.

In [ ]:
all_appear = pd.concat([ep["a0"], ep["a1"]]).value_counts()
noex = all_appear.get("no ex ace", 0)
print(f"sides with no ex ace: {noex} of {all_appear.sum()} ({noex/all_appear.sum()*100:.1f}%), "
      "shown here once and excluded from the rankings below")
appear = all_appear[all_appear.index != "no ex ace"]
share = (appear / appear.sum() * 100)
top = share.head(14)[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(range(len(top)), top.values, color=INK, alpha=0.85)
ax.set_yticks(range(len(top)), [t[:34] for t in top.index])
ax.set_xlabel("share of ladder appearances (%)")
ax.set_title("Archetype usage on the ladder", color=INK, loc="left", fontweight="bold")
for i, v in enumerate(top.values): ax.text(v + 0.1, i, f"{v:.1f}", va="center", fontsize=8, color=MUT)
plt.tight_layout(); plt.show()
KEEP = appear[appear >= max(20, appear.quantile(0.5))].index.tolist()   # archetypes with enough games
print("archetypes with a workable sample:", len(KEEP))

## 2. The tier list, by win rate with honest error bars

Usage is popularity, not strength. Here is each archetype's overall win rate (a side counts once per
battle it is in), with a Wilson interval so you can see which gaps are real and which are noise.

In [ ]:
def side_records(arch):
    w = ((ep.a0 == arch) & (ep.winner == 0)).sum() + ((ep.a1 == arch) & (ep.winner == 1)).sum()
    n = (ep.a0 == arch).sum() + (ep.a1 == arch).sum()
    return w, n

def wilson(w, n, z=1.96):
    if n == 0: return (0, 0, 0)
    p = w / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return p, c-h, c+h

rec = []
for a in KEEP:
    w, n = side_records(a); p, lo, hi = wilson(w, n)
    rec.append(dict(archetype=a, games=n, winrate=p, lo=lo, hi=hi))
T = pd.DataFrame(rec).sort_values("winrate", ascending=True)
fig, ax = plt.subplots(figsize=(9, max(4, 0.42*len(T))))
ax.errorbar(T.winrate*100, range(len(T)), xerr=[(T.winrate-T.lo)*100, (T.hi-T.winrate)*100],
            fmt="o", color=INK, ecolor=MUT, capsize=3)
ax.axvline(50, color=ACC, lw=1, ls="--")
ax.set_yticks(range(len(T)), [f"{a[:30]} (n={n})" for a, n in zip(T.archetype, T.games)])
ax.set_xlabel("win rate (%)  with 95% Wilson interval")
ax.set_title("Ladder tier list: win rate, not popularity", color=INK, loc="left", fontweight="bold")
plt.tight_layout(); plt.show()
print(T.sort_values("winrate", ascending=False).round(3).to_string(index=False))

## 3. The matchup grid

Overall win rate blends easy and hard matchups. This grid is the real structure: row archetype's win
rate against column archetype. Cells with too few games are left blank so you are not misled by a
single lucky pairing.

In [ ]:
TOPK = T.sort_values("games", ascending=False).head(8)["archetype"].tolist()
M = np.full((len(TOPK), len(TOPK)), np.nan); N = np.zeros_like(M)
for i, ra in enumerate(TOPK):
    for j, ca in enumerate(TOPK):
        if i == j: continue
        m = ((ep.a0 == ra) & (ep.a1 == ca))
        m2 = ((ep.a0 == ca) & (ep.a1 == ra))
        w = (m & (ep.winner == 0)).sum() + (m2 & (ep.winner == 1)).sum()
        n = m.sum() + m2.sum()
        if n >= 8: M[i, j] = w / n; N[i, j] = n
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(M*100, cmap="RdBu_r", vmin=25, vmax=75, aspect="auto")
ax.set_xticks(range(len(TOPK)), [a[:16] for a in TOPK], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(TOPK)), [a[:22] for a in TOPK], fontsize=8)
for i in range(len(TOPK)):
    for j in range(len(TOPK)):
        if not np.isnan(M[i, j]):
            ax.text(j, i, f"{M[i,j]*100:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(M[i,j]-0.5) > 0.18 else INK)
ax.set_title("Matchup win rate: row vs column (%)", color=INK, loc="left", fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.046, label="row win rate (%)")
plt.tight_layout(); plt.show()

## 4. Which decks end games fast

Game length matters here for a mundane reason: the harness runs on a clock, and an agent that thinks
long in a deck that grinds long games is the one that times out. Step count in the log is a rough but
consistent proxy for game length, so here is how fast each archetype's games end, split by win and loss.

In [ ]:
L = []
for a in TOPK:
    m0 = (ep.a0 == a); m1 = (ep.a1 == a)
    won = pd.concat([ep[m0 & (ep.winner == 0)].n_steps, ep[m1 & (ep.winner == 1)].n_steps])
    lost = pd.concat([ep[m0 & (ep.winner == 1)].n_steps, ep[m1 & (ep.winner == 0)].n_steps])
    L.append(dict(archetype=a, med_steps_win=won.median(), med_steps_loss=lost.median(),
                  med_steps=pd.concat([won, lost]).median(), n=len(won)+len(lost)))
LD = pd.DataFrame(L).sort_values("med_steps")
fig, ax = plt.subplots(figsize=(9, 4.8))
x = np.arange(len(LD)); w = 0.4
ax.bar(x - w/2, LD.med_steps_win, w, label="in wins", color=ACC2)
ax.bar(x + w/2, LD.med_steps_loss, w, label="in losses", color=MUT)
ax.set_xticks(x, [a[:14] for a in LD.archetype], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("median steps per game")
ax.set_title("Fast finishers and grinders, by archetype", color=INK, loc="left", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()
print(LD.round(1).to_string(index=False))

## 5. Does popularity track strength?

If everyone were rational the most-played decks would be the strongest. The chart below shows how
true that is on this day. Decks in the lower right are over-played for their results (crowd
favourites that underperform); the upper left is where the quiet value sits, strong decks that are
still under-played, and the printout underneath names any that qualify. That corner is exactly where
a submission can steal rating.

In [ ]:
sc = T.merge(share.rename("usage").reset_index().rename(columns={"index": "archetype"}), on="archetype", how="left")
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.scatter(sc.usage, sc.winrate*100, s=np.sqrt(sc.games)*7, color=ACC2, alpha=0.75, edgecolor="white")
ax.axhline(50, color=MUT, lw=1, ls="--")
for _, r in sc.iterrows():
    if r.usage > sc.usage.median() or abs(r.winrate-0.5) > 0.05:
        ax.annotate(r.archetype[:18], (r.usage, r.winrate*100), fontsize=7.5, color=INK,
                    xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("usage share (%)"); ax.set_ylabel("win rate (%)")
ax.set_title("Under-played and winning is where the rating is", color=INK, loc="left", fontweight="bold")
corr = np.corrcoef(sc.usage, sc.winrate)[0, 1]
ax.text(0.98, 0.03, f"usage vs win-rate corr = {corr:+.2f} (few points, indicative only)",
        transform=ax.transAxes, ha="right", color=MUT)
plt.tight_layout(); plt.show()
under = sc[(sc.winrate > 0.52) & (sc.usage < sc.usage.median())].sort_values("winrate", ascending=False)
print("strong but under-played this day:")
if len(under):
    print(under[["archetype", "usage", "winrate", "lo", "hi", "games"]].round(3).to_string(index=False))
    if (under.lo <= 0.5).any():
        print("note: where the interval crosses 50%, read these as leads to test, not conclusions")
else:
    print("  none this day; the crowd is playing the winners")

## 6. So which deck should you actually submit

The tier list and the matchup grid answer different questions, and neither is the one you care about
when you pick a deck. What you want is the expected win rate against the field you will actually face,
which is each deck's matchup results weighted by how often each opponent shows up. That collapses
everything above into one number per deck, and it is computed here from this day's meta, so it moves
when the meta moves. Alongside it is the sharpest answer to whatever deck is currently over-played,
since punishing the crowd's favourite is the cheapest rating on the ladder.

In [ ]:
usage_k = appear[KEEP] / appear[KEEP].sum()
def mwr(mine, opp):
    m = (ep.a0 == mine) & (ep.a1 == opp); m2 = (ep.a0 == opp) & (ep.a1 == mine)
    w = (m & (ep.winner == 0)).sum() + (m2 & (ep.winner == 1)).sum(); n = m.sum() + m2.sum()
    return (w / n, n) if n >= 8 else (None, n)

recs = []
for mine in KEEP:
    num = den = 0.0
    for opp in KEEP:
        wr, _ = mwr(mine, opp)
        if wr is not None:
            num += usage_k[opp] * wr; den += usage_k[opp]
    if den > 0:
        recs.append(dict(deck=mine, exp_vs_field=num/den, coverage=den, games=int(appear[mine])))
RC = pd.DataFrame(recs).sort_values("exp_vs_field", ascending=True)
fig, ax = plt.subplots(figsize=(9, max(3.5, 0.45*len(RC))))
colors = [ACC if v > 0.5 else MUT for v in RC.exp_vs_field]
ax.barh(range(len(RC)), RC.exp_vs_field*100, color=colors)
ax.axvline(50, color=INK, lw=1, ls="--")
ax.set_yticks(range(len(RC)), [f"{d[:26]} ({c*100:.0f}% cov)" for d, c in zip(RC.deck, RC.coverage)])
ax.set_xlabel("expected win rate against the field (%)")
ax.set_title("Best deck to bring into this field", color=INK, loc="left", fontweight="bold")
for i, v in enumerate(RC.exp_vs_field*100): ax.text(v+0.2, i, f"{v:.1f}", va="center", fontsize=8, color=MUT)
plt.tight_layout(); plt.show()

best = RC.iloc[-1]
print(f"meta pick: {best.deck}  ->  {best.exp_vs_field*100:.1f}% expected vs the field "
      f"(covers {best.coverage*100:.0f}% of the ranked field)")
mp = usage_k.idxmax()
cts = sorted([(o, mwr(o, mp)[0]) for o in KEEP if o != mp and mwr(o, mp)[0] is not None],
             key=lambda x: -x[1])[:3]
print(f"most-played is {mp} at {usage_k.max()*100:.0f}%; sharpest counters to it:")
for o, wr in cts:
    print(f"   {o[:28]:28s} {wr*100:.0f}% into {mp[:22]}")
print("coverage is the share of the ranked field this deck has a measured matchup against; "
      "a low number means the estimate leans on fewer pairings.")

## What moved since last week

A snapshot tells you where the field stands, not where it is heading. This section reads the last
published snapshot, if it is attached, and compares it against this run, so you can see which
archetypes gained or lost ladder share and whose win rate actually moved. A win-rate move is called
real only when the two Wilson intervals do not overlap, so a small wobble on a sampled day is not
read as a trend. Attach the maintained snapshot dataset to light this up; without it the section
prints a short note and the rest of the notebook is unaffected.

In [ ]:
# This run differenced against the last published snapshot.
# Attach busyaprime/pokemon-tcg-ai-battle-live-meta so its tier_and_usage.csv reads as "previous".
_pm = re.search(r"(\d{4}-\d{2}-\d{2})", epdir or "")
cur_day = _pm.group(1) if _pm else "unknown"
prev_path = find_one(["tier_and_usage.csv"])
prev = pd.read_csv(prev_path) if prev_path else None
if prev is None or "day" not in getattr(prev, "columns", []):
    print("no previous snapshot found; attach pokemon-tcg-ai-battle-live-meta to see week-over-week movement")
elif str(prev["day"].iloc[0]) == cur_day:
    print(f"previous snapshot is the same day ({cur_day}); nothing to compare")
else:
    prev_day = str(prev["day"].iloc[0])
    D = sc.merge(prev, on="archetype", how="outer", suffixes=("_cur", "_prev"), indicator=True)
    both = D[D["_merge"] == "both"].copy()
    both["d_usage"] = both.usage_cur - both.usage_prev
    both["d_winrate"] = both.winrate_cur - both.winrate_prev
    both["real"] = (both.lo_cur > both.hi_prev) | (both.lo_prev > both.hi_cur)
    both = both.sort_values("d_usage")
    new = sorted(D.loc[D["_merge"] == "left_only", "archetype"])
    gone = sorted(D.loc[D["_merge"] == "right_only", "archetype"])
    if len(both):
        fig, ax = plt.subplots(figsize=(9, max(3.5, 0.45*len(both))))
        ax.barh(range(len(both)), both.d_usage, color=[ACC if v > 0 else ACC2 for v in both.d_usage])
        ax.axvline(0, color=INK, lw=1)
        ax.set_yticks(range(len(both)), [a[:30] for a in both.archetype])
        ax.set_xlabel(f"change in ladder share since {prev_day} (percentage points)")
        ax.set_title("What moved: usage change vs the last snapshot", color=INK, loc="left", fontweight="bold")
        plt.tight_layout(); plt.show()
        mv = both.assign(
            usage_prev=both.usage_prev.round(1), usage_cur=both.usage_cur.round(1),
            d_usage=both.d_usage.round(1), winrate_prev=(both.winrate_prev*100).round(1),
            winrate_cur=(both.winrate_cur*100).round(1), d_winrate=(both.d_winrate*100).round(1))
        print(f"movement {prev_day} -> {cur_day} (share and win rate in %):")
        print(mv[["archetype", "usage_prev", "usage_cur", "d_usage",
                  "winrate_prev", "winrate_cur", "d_winrate", "real"]].to_string(index=False))
        print("real=True means the two win-rate intervals do not overlap; "
              "small share moves on a sampled day are noise, not a trend")
    if new:  print("new to the tracked set this week:", ", ".join(map(str, new)))
    if gone: print("left the tracked set (rarer, or below this run's sample threshold):", ", ".join(map(str, gone)))

## Honest scope

This is one day of ladder battles, so treat it as a snapshot, not a law; fork it onto another day to
see what moves. On a busy day I read a random sample of at most 4000 episode files (fixed seed), not
necessarily the full dump; the parse cell prints exactly how many games went in. The archetype label
is a heuristic, a side's highest-HP ex-Pokemon that it actually played; a deck built around a non-ex
attacker lands in a catch-all bucket that the parse reports once but the rankings exclude, and in a
small share of games the highest-HP ex is a tech card rather than the deck's true engine. Ties,
crashed games and malformed logs are dropped. Step count is a rough but consistent proxy for game
length, not a turn counter. Everything here is recomputed from the raw logs when you run the
notebook, with no pasted numbers and no leaderboard claim, so you can trust it exactly as far as the
parse, which is all shown above.

## Take the tables with you

The cell below writes the computed meta out as tidy CSVs, so you can fork this notebook, or attach its
output as a dataset, and load the tier list, the matchup grid and the deck recommendation without parsing
the full log dump again. Every file is stamped with the day it was computed from. A maintained snapshot of these tables is also
kept as a dataset if you would rather just download them:
[pokemon-tcg-ai-battle-live-meta](https://www.kaggle.com/datasets/busyaprime/pokemon-tcg-ai-battle-live-meta).

In [ ]:
import os, re
import pandas as pd
_m = re.search(r"(\d{4}-\d{2}-\d{2})", epdir or "")
DAY = _m.group(1) if _m else "unknown"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
sc.assign(day=DAY).to_csv(f"{OUT}/tier_and_usage.csv", index=False)       # archetype, games, winrate, lo, hi, usage
LD.assign(day=DAY).to_csv(f"{OUT}/game_length.csv", index=False)          # archetype, med_steps_win/loss/med
RC.assign(day=DAY).to_csv(f"{OUT}/deck_recommender.csv", index=False)     # deck, exp_vs_field, coverage, games
mg = pd.DataFrame(M, index=list(TOPK), columns=list(TOPK)); mg.index.name = "deck_row_winrate_vs_col"
mg.to_csv(f"{OUT}/matchup_grid_winrate.csv")
pd.DataFrame(N, index=list(TOPK), columns=list(TOPK)).to_csv(f"{OUT}/matchup_grid_games.csv")
print("wrote meta CSVs for", DAY, "->", sorted(f for f in os.listdir(OUT) if f.endswith(".csv")))